PhysicsBody.java

In [ ]:
import greenfoot.*; 

public abstract class PhysicsBody extends Actor
{
    private double exactX;
    private double exactY;

    @Override
    public void move(int distance)
    {
        move((double)distance);
    }
    
    public void move(double distance)
    {
        double radians = Math.toRadians(getRotation());
        double dx = Math.cos(radians) * distance;
        double dy = Math.sin(radians) * distance;
        setLocation(exactX + dx, exactY + dy);
    }
    
    public void setLocation(double x, double y) 
    {
        exactX = x;
        exactY = y;
        super.setLocation((int) (x + 0.5), (int) (y + 0.5));
    }
    
    @Override
    public void setLocation(int x, int y) 
    {
        exactX = x;
        exactY = y;
        super.setLocation(x, y);
    }

    public double getExactX() 
    {
        return exactX;
    }

    public double getExactY() 
    {
        return exactY;
    }
}


# BÁO CÁO CẤU TRÚC LỚP PHYSICSBODY - GAME PROJECT
---

## 1. Giới thiệu chung
Lớp `PhysicsBody` là một lớp trừu tượng (`abstract class`) đóng vai trò xử lý **Tọa độ chính xác (Smooth Mobility)** cho tất cả các đối tượng trong game. Thay vì sử dụng hệ tọa độ số nguyên mặc định của Greenfoot, lớp này cho phép các vật thể di chuyển với độ chính xác số thập phân, loại bỏ hiện tượng rung lắc hình ảnh khi di chuyển ở tốc độ thấp.

---

## 2. Các thuộc tính cốt lõi
Lớp sử dụng hai biến riêng tư để quản lý vị trí thực tế của vật thể:

| Thuộc tính | Kiểu dữ liệu | Mô tả |
| :--- | :--- | :--- |
| `exactX` | `double` | Lưu trữ tọa độ X thực tế với độ chính xác số thập phân. |
| `exactY` | `double` | Lưu trữ tọa độ Y thực tế với độ chính xác số thập phân. |

---

## 3. Các phương thức quan trọng

### 3.1. Phương thức `setLocation` (Overloading)
Đây là phần quan trọng nhất của lớp. Nó ghi đè phương thức mặc định của Greenfoot để xử lý dữ liệu:
* **Ghi nhớ tọa độ thực:** Lưu giá trị vào `exactX` và `exactY`.
* **Làm tròn hiển thị:** Sử dụng công thức `(int) (x + 0.5)` để làm tròn tọa độ về số nguyên gần nhất trước khi đưa lên màn hình Greenfoot, giúp hình ảnh không bị hiển thị sai lệch.

### 3.2. Phương thức `move` (Di chuyển theo góc)
Hàm này sử dụng lượng giác học để tính toán hướng di chuyển dựa trên góc quay (`Rotation`) của vật thể:

$$dx = \cos(\text{radians}) \cdot \text{distance}$$
$$dy = \sin(\text{radians}) \cdot \text{distance}$$



---

## 4. Giải thuật toán học
Để di chuyển vật thể một cách chính xác trong không gian 2D, lớp sử dụng các phép toán lượng giác sau:

1.  **Chuyển đổi góc:** Chuyển đổi góc từ Độ (Degrees) sang Radian để máy tính có thể tính toán:
    $$\text{radians} = \frac{\pi \cdot \text{angle}}{180}$$
2.  **Cập nhật vị trí:**
    $$x_{new} = x_{old} + \cos(\theta) \cdot d$$
    $$y_{new} = y_{old} + \sin(\theta) \cdot d$$

> **Ưu điểm:** Phương pháp này cho phép nhân vật di chuyển theo bất kỳ hướng nào (360 độ) mà không bị giới hạn bởi các ô vuông tọa độ nguyên.



---

## 5. Kết luận
Lớp `PhysicsBody` là nền tảng kỹ thuật thiết yếu cho các lớp kế thừa như `SpriteAnimator` và `Plant`. Nó đảm bảo mọi chuyển động trong trò chơi, từ việc cây rung rinh đến việc đạn bay hay Zombie đi bộ, đều diễn ra một cách **mượt mà (smooth)** và ổn định về mặt logic toán học.

\newpage

In [ ]:
import greenfoot.*; 
import java.util.Timer;
import java.io.*;

public class SpriteAnimator extends PhysicsBody
{
    
    public long deltaTime;
    public long lastFrame = System.nanoTime();
    public long currentFrame = System.nanoTime();
    public GreenfootImage[] previousSprites = null;
    
    public int frame = 0;
    
    public GreenfootImage[] importSprites(String filename, int frames) {
        GreenfootImage[] temp = new GreenfootImage[frames];
        for (int i = 0; i < frames; i++) {
            GreenfootImage scale = new GreenfootImage(filename+(i+1)+".png");
            
            temp[i] = scale;
        }
        return temp;
    }
    
    public GreenfootImage[] importSprites(String filename, int frames, double scaleFactor) {
        GreenfootImage[] temp = new GreenfootImage[frames];
        for (int i = 0; i < frames; i++) {
            GreenfootImage scale = new GreenfootImage(filename+(i+1)+".png");
            scale.scale((int)(scale.getWidth()*scaleFactor), (int)(scale.getHeight()*scaleFactor));
            temp[i] = scale;
        }
        return temp;
    }
    
    
    
    public void animate(GreenfootImage[] sprite, long duration) {
        currentFrame = System.nanoTime();
        deltaTime = (currentFrame - lastFrame) / 1000000;
        
        
        if (deltaTime > duration) {
            lastFrame = currentFrame;    
            frame++;
            if (frame < sprite.length) {
                setImage(sprite[frame]);
            }

        }
        if (frame < sprite.length) {
            
        } else {
            
            frame = 0;
            setImage(sprite[frame]);
        }
        if (!sprite.equals(previousSprites)) {
            setImage(sprite[frame]);
            previousSprites = sprite;
        }
        
           
    }
    
    public void animate(GreenfootImage[] sprite, long duration, boolean loop) {
        currentFrame = System.nanoTime();
        deltaTime = (currentFrame - lastFrame) / 1000000;
           
        if (deltaTime > duration) {
            
            lastFrame = currentFrame;    
            frame++;
            if (frame < sprite.length) {
                setImage(sprite[frame]);
            }
        }
        if (frame < sprite.length) {
            
        } else if (loop) {
            
            frame = 0;
            setImage(sprite[frame]);
        } else {
            
        }
        if (!sprite.equals(previousSprites)) {
            if (frame < sprite.length) {
                setImage(sprite[frame]);
                previousSprites = sprite;
            }
        }
               
        

        
    }
    public void setFrame(int toFrame) {
        frame = toFrame-1;
    
    }
    
    public int getCurrentFrame() {
        return frame+1;
    }
    public void hitFlash(GreenfootImage[] sprite, String filename) {
        int tempFrame = frame;
        int flashFrame = tempFrame+1;
        if (tempFrame >= sprite.length) {
            tempFrame = 0;
            flashFrame = 1;
          
        } else if (flashFrame >= sprite.length) {
            flashFrame = 0;
        }
      
        
        
        GreenfootImage first = new GreenfootImage("flash"+filename+(tempFrame+1)+".png");
        GreenfootImage second = new GreenfootImage("flash"+filename+(flashFrame+1)+".png");
 
        sprite[tempFrame] = first;
        sprite[flashFrame] = second;
        
        
        setImage(sprite[tempFrame]);
        
        Timer timer = new Timer();
        timer.schedule(new Timer1(sprite, filename, tempFrame,flashFrame), 500L); 
       
    }

}


# BÁO CÁO CẤU TRÚC LỚP SPRITEANIMATOR - GAME PROJECT
---

## 1. Giới thiệu chung
Lớp `SpriteAnimator` kế thừa từ `PhysicsBody`, đóng vai trò là **Bộ khung xử lý hoạt ảnh (Animation Engine)** cho các đối tượng trong trò chơi. Lớp này giải quyết vấn đề cốt lõi trong phát triển game: Đảm bảo tốc độ hoạt ảnh diễn ra đồng nhất dựa trên thời gian thực, thay vì phụ thuộc vào tốc độ khung hình (FPS) biến thiên của phần cứng.

---

## 2. Quản lý Thời gian thực (Delta Time Logic)
Để hoạt ảnh mượt mà, lớp sử dụng hệ thống tính toán khoảng chênh lệch thời gian:

| Thành phần | Kiểu dữ liệu | Ý nghĩa kỹ thuật |
| :--- | :--- | :--- |
| `lastFrame` | `long` | Lưu thời điểm (theo nano giây) của lần cập nhật trước. |
| `deltaTime` | `long` | Khoảng thời gian đã trôi qua tính bằng mili giây (ms). |
| `duration` | `long` | Tham số quyết định tốc độ của hoạt ảnh (ms mỗi khung hình). |

---

## 3. Các tính năng chính

### 3.1. Nhập và Xử lý tài nguyên (`importSprites`)
Hàm này hỗ trợ nạp tự động danh sách hình ảnh từ ổ cứng vào mảng `GreenfootImage[]`. 
* **Hỗ trợ Scaling:** Cho phép thay đổi kích thước toàn bộ bộ Sprite ngay khi nạp vào RAM để tối ưu hóa hiệu năng render.
* **Cấu trúc tên file:** Mặc định nhận diện theo định dạng `tên + số thứ tự.png`.

### 3.2. Điều khiển hoạt ảnh (`animate`)
Lớp cung cấp hai phương thức nạp chồng (Overloading) để xử lý hoạt ảnh:
1.  **Chế độ mặc định:** Tự động lặp lại (Loop) khi đi đến cuối mảng hình ảnh.
2.  **Chế độ tùy chỉnh:** Cho phép chọn giữa lặp lại liên tục hoặc dừng lại ở khung hình cuối cùng (thường dùng cho các hiệu ứng nổ hoặc chết).



### 3.3. Hiệu ứng phản hồi thị giác (`hitFlash`)
Đây là một kỹ thuật nâng cao giúp tăng "Game Feel". Khi đối tượng bị trúng đòn:
* Hệ thống tạm thời thay thế các khung hình hiện tại bằng phiên bản "Flash" (thường là màu trắng hoàn toàn).
* Sử dụng `java.util.Timer` để tự động khôi phục lại hình ảnh gốc sau một khoảng thời gian ngắn (500ms).

---

## 4. Cơ chế hoạt động của thuật toán Animation
Thuật toán kiểm tra điều kiện chuyển khung hình dựa trên thời gian trôi qua:

$$\text{Nếu } (t_{\text{current}} - t_{\text{last}}) \geq \text{duration} \implies \text{Chuyển khung hình tiếp theo}$$

Lớp cũng lưu trữ biến `previousSprites` để nhận diện sự thay đổi trạng thái hành động (ví dụ: Chuyển từ "Đi bộ" sang "Tấn công"). Nếu bộ ảnh thay đổi, chỉ số `frame` sẽ lập tức được reset về 0 để tránh lỗi hiển thị sai lệch khung hình.



---

## 5. Kết luận
Lớp `SpriteAnimator` là cầu nối quan trọng giữa logic vật lý và hiển thị đồ họa. Nhờ việc quản lý hoạt ảnh theo thời gian thực và hỗ trợ các hiệu ứng hình ảnh linh hoạt, nó giúp nhân vật trong game có những chuyển động sống động, chuyên nghiệp và phản hồi tốt với tương tác của người chơi.

\newpage

Plant.java

- Class Plant: Lớp cha cho tất cả các loại cây trong game (Sunflower, Peashooter, v.v.)
- Kế thừa SpriteAnimator để có khả năng chạy hoạt ảnh mượt mà.

In [ ]:
import greenfoot.*;  

public class Plant extends SpriteAnimator
{
    // --- THÔNG SỐ CƠ BẢN CỦA CÂY ---
    public int cost;      // Giá tiền để mua cây
    public int maxHp;     // Máu tối đa
    public int hp;        // Máu hiện tại
    public int damage;    // Sát thương (nếu cây có tấn công)
    public boolean isAlive = true; // Trạng thái còn sống hay không
    public boolean opaque = false; // Trạng thái làm mờ cây (ví dụ khi đang chọn vị trí)
    
    // Tham chiếu đến thế giới chơi game để gọi các hàm điều khiển bàn cờ
    public PlayScene PlayScene; 

    // --- LOGIC KÉO THẢ (DRAG & DROP) ---
    private boolean isDragging = false; // Cây có đang bị người chơi kéo đi không
    private int startGridX, startGridY; // Tọa độ ô lưới cũ trước khi kéo

    public Plant() {
    }

    /**
     * VÒNG LẶP CHÍNH (ACT): Chạy liên tục mỗi khung hình
     */
    public void act()
    {
        PlayScene world = (PlayScene) getWorld();
        if (world == null) return; // Nếu cây chưa vào thế giới thì thoát
        
        // Nếu có bảng Menu (Overlay) đang hiện ra thì tạm dừng mọi hoạt động của cây
        if (!world.getObjects(Overlay.class).isEmpty()) {
            return; 
        }
        
        // Gọi hàm xử lý tương tác chuột (kéo thả)
        handleMouse();

        // KIỂM TRA SINH MỆNH
        if (isLiving()) {
            update(); // Hàm này để các lớp con (ví dụ Peashooter) viết code tấn công riêng
            
            // XỬ LÝ ĐỘ TRONG SUỐT (Visual Feedback)
            if (world != null && getImage() != null) {
                if (isDragging) {
                    getImage().setTransparency(125); // Đang kéo thì làm mờ 50%
                } else if (!opaque) {
                    getImage().setTransparency(255); // Bình thường thì hiện rõ 100%
                } else {
                    getImage().setTransparency(125); // Nếu bị ép opaque thì làm mờ
                }
            }
        } else {
            // LOGIC KHI CÂY CHẾT (Bị Zombie ăn hết máu)
            PlayScene = (PlayScene)getWorld();
            AudioPlayer.play(80, "gulp.mp3"); // Phát âm thanh "ngồm ngoàm"
            
            int x = getXPos(); // Lấy vị trí cột hiện tại
            int y = getYPos(); // Lấy vị trí hàng hiện tại
            
            // Xóa cây khỏi dữ liệu của bàn cờ (Board)
            if (PlayScene.board != null) {
                PlayScene.board.removePlant(x, y);
            }
            // Xóa cây khỏi màn hình game
            PlayScene.removeObject(this);
            return;
        }
    }

    /**
     * HÀM XỬ LÝ CHUỘT: Cho phép cầm cây di chuyển sang ô khác
     */
    private void handleMouse() {
        MouseInfo mouse = Greenfoot.getMouseInfo();
    
        // 1. Khi bắt đầu nhấn chuột vào cây (Chuột trái - Button 1)
        if (Greenfoot.mousePressed(this)) {
            if (mouse != null && mouse.getButton() == 1) { 
                isDragging = true;
                startGridX = getXPos(); // Lưu lại vị trí cột ban đầu
                startGridY = getYPos(); // Lưu lại vị trí hàng ban đầu
                
                // Đẩy cây lên lớp trên cùng để không bị đè bởi bàn cờ khi kéo
                if (PlayScene != null) {
                    PlayScene.setPaintOrder(Plant.class, Board.class);
                }
            }
        }
    
        // 2. Khi đang giữ và kéo chuột đi
        if (isDragging && Greenfoot.mouseDragged(this)) {
            if (mouse != null) {
                setLocation(mouse.getX(), mouse.getY()); // Hình ảnh cây đi theo con trỏ chuột
            }
        }
    
        // 3. Khi thả chuột ra
        if (isDragging && Greenfoot.mouseDragEnded(this)) {
            isDragging = false;
            
            int newGridX = getXPos(); // Tính xem người chơi định thả vào cột nào
            int newGridY = getYPos(); // Tính xem người chơi định thả vào hàng nào
    
            // Gửi lệnh di chuyển đến Board. Nếu ô mới trống (movePlant trả về true):
            if (PlayScene.board != null && PlayScene.board.movePlant(startGridX, startGridY, newGridX, newGridY, this)) {
                // Đặt cây vào chính tâm của ô lưới mới
                setLocation(PlayScene.board.getXCoord(newGridX, newGridY), PlayScene.board.getYCoord(newGridX, newGridY));
            } else {
                // Nếu ô mới đã có cây, trả cây về vị trí cũ ban đầu
                setLocation(PlayScene.board.getXCoord(startGridX, startGridY), PlayScene.board.getYCoord(startGridX, startGridY));
            }
        }
    }

    /**
     * HÀM TOÁN HỌC: Chuyển tọa độ Pixel (X) sang chỉ số Cột (0-8)
     */
    public int getXPos() {
        if (getWorld() == null || PlayScene.board == null) return 0;
        Board b = PlayScene.board;
        
        // Công thức tính ma trận để tìm tọa độ trong không gian nghiêng (Isometric/Perspective)
        double det = (double)(b.xSpacing * b.ySpacing) - (b.rowDeltaX * b.colDeltaY);
        double dx = getX() - b.xOffset;
        double dy = getY() - b.yOffset;
        
        int x = (int)Math.round((dx * b.ySpacing - dy * b.rowDeltaX) / det);
        
        // Giới hạn trong phạm vi bàn cờ (9 cột)
        if (x < 0) return 0;
        if (x > 8) return 8;
        return x;
    }

    /**
     * HÀM TOÁN HỌC: Chuyển tọa độ Pixel (Y) sang chỉ số Hàng (0-4)
     */
    public int getYPos() {
        if (getWorld() == null || PlayScene.board == null) return 0;
        Board b = PlayScene.board;
        
        double det = (double)(b.xSpacing * b.ySpacing) - (b.rowDeltaX * b.colDeltaY);
        double dx = getX() - b.xOffset;
        double dy = getY() - b.yOffset;
        
        int y = (int)Math.round((b.xSpacing * dy - b.colDeltaY * dx) / det);
        
        // Giới hạn trong phạm vi bàn cờ (5 hàng)
        if (y < 0) return 0;
        if (y > 4) return 4; 
        return y;
    }

    /**
     * HÀM TỰ ĐỘNG CHẠY KHI CÂY ĐƯỢC ĐẶT XUỐNG WORLD
     */
    @Override
    public void addedToWorld(World world) {
        PlayScene = (PlayScene)world;
        // Tạo thêm hiệu ứng bụi đất (Dirt) ở dưới chân cây cho đẹp
        PlayScene.addObject(new Dirt(), getX(), getY() + 30);
    }

    public void update() {
        // Sẽ được override ở các class con như Sunflower, Peashooter...
    }
    
    public void activatePlantFood() {                                     
        this.hp = maxHp; // Khi dùng Plant Food, cây sẽ được hồi đầy máu
    }

    public boolean isLiving() {
        return hp > 0; // Cây còn sống nếu máu > 0
    }

    public void hit(int dmg) {
        hp -= dmg; // Bị trúng đòn thì trừ máu
    }
}

# BÁO CÁO CẤU TRÚC LỚP PLANT - GAME PROJECT
---

## 1. Giới thiệu chung
Lớp `Plant` là một lớp trừu tượng (abstract-like) đóng vai trò là lớp cha (Base Class) cho tất cả các đơn vị thực vật trong trò chơi. Lớp này kế thừa từ `SpriteAnimator`, giúp thừa hưởng các khả năng:
* **Xử lý hoạt ảnh mượt mà** dựa trên thời gian thực (Delta Time).
* **Quản lý trạng thái vật lý** cơ bản thông qua lớp `PhysicsBody`.



---

## 2. Các thuộc tính quan trọng (Properties)
Dưới đây là các thông số kỹ thuật cốt lõi được định nghĩa trong lớp:

| Thuộc tính | Kiểu dữ liệu | Mô tả |
| :--- | :--- | :--- |
| **cost** | `int` | Lượng tài nguyên cần thiết để trồng cây. |
| **hp / maxHp** | `int` | Chỉ số máu hiện tại và giới hạn chịu đựng của thực vật. |
| **isDragging** | `boolean` | Xác định trạng thái cây đang được kéo thả bởi chuột. |
| **PlayScene** | `Object` | Tham chiếu đến thế giới chính để điều khiển bàn cờ (`Board`). |
| **opaque** | `boolean` | Hiệu ứng hình ảnh (mờ/rõ) phục vụ phản hồi thị giác. |

---

## 3. Các phương thức cốt lõi (Core Methods)

### 3.1. Vòng đời và Cập nhật (`act`)
Hàm `act()` đóng vai trò là "bộ não" điều khiển, kiểm tra điều kiện sống còn mỗi khung hình. Quy trình hủy đối tượng khi `hp <= 0` diễn ra như sau:
1.  **Audio:** Phát tệp âm thanh `gulp.mp3` thông qua `AudioPlayer`.
2.  **Data:** Gọi `board.removePlant(x, y)` để giải phóng ô lưới.
3.  **Memory:** Tự xóa khỏi thế giới (`removeObject`) để tối ưu bộ nhớ.

### 3.2. Hệ thống kéo thả (Drag & Drop)
Phương thức `handleMouse()` quản lý tương tác người dùng qua 3 giai đoạn logic:
* **Giai đoạn Pressed:** Kích hoạt `isDragging`, ghi nhớ tọa độ lưới cũ (`startGridX`, `startGridY`).
* **Giai đoạn Dragged:** Liên tục cập nhật tọa độ Pixel của Object theo `MouseInfo`.
* **Giai đoạn Ended:** Kiểm tra tính hợp lệ của vị trí mới. Nếu ô đã có cây, thực hiện logic "Snap-back" về vị trí cũ.



<div style="page-break-after: always;"></div>

## 4. Giải thuật tính toán tọa độ lưới
Đây là thành phần quan trọng nhất giúp chuyển đổi từ tọa độ màn hình (**Pixels**) sang tọa độ bàn cờ (**Grid Indices**). 

Vì bàn cờ sử dụng góc nhìn phối cảnh (Isometric), chúng ta không thể dùng phép chia đơn thuần. Giải thuật sử dụng **Ma trận nghịch đảo** để ánh xạ tọa độ:

$$x = \text{round} \left( \frac{dx \cdot ySpacing - dy \cdot rowDeltaX}{det} \right)$$

$$y = \text{round} \left( \frac{xSpacing \cdot dy - colDeltaY \cdot dx}{det} \right)$$

> **Lưu ý:** Biến `det` đại diện cho định thức (determinant) của ma trận biến đổi không gian bàn cờ, giúp đảm bảo tính chính xác khi người chơi tương tác tại bất kỳ vị trí nào trên lưới 9x5.



---

## 5. Kết luận
Lớp `Plant` không chỉ quản lý tốt các thuộc tính sinh học của thực vật mà còn giải quyết triệt để các vấn đề về tương tác người dùng và toán học không gian. Đây là nền tảng vững chắc để mở rộng cho các lớp con cụ thể như `Peashooter`, `Sunflower` hay `Wall-nut`.

\newpage

In [ ]:
import greenfoot.*;
import java.util.List;

public class BonkChoy extends Plant {
    private GreenfootImage[] idle, pRight, pfStart, pfLoop, pfEnd, kRight;
    private GreenfootImage pfTransformImg;
    private GreenfootImage deathBg;
    private GreenfootImage originalWorldBg;
    
    private int pfStage = 0, frameIndex = 0, punchCount = 0;
    private boolean adjusted = false, isUsingPF = false, pfSoundPlayed = false, bgActive = false;
    private long lastAttackTime = System.currentTimeMillis(), pfTimer = 0, lastFrameTime = 0;

    public BonkChoy() {
        maxHp = 670; 
        hp = maxHp;
        
        kRight = loadResized("bonkchoyknockoutone", 15);
        idle = loadResized("bonkchoyidle_three", 23);
        pRight = loadResized("bonkchoyattackone", 10);
        pfStart = loadResized("bonkchoyplantfoodstart", 30);
        pfLoop = loadResized("bonkchoyplantfood", 30);
        pfEnd = loadResized("bonkchoyplantfoodend", 10);
        
        pfTransformImg = new GreenfootImage("bonkchoyplantfood.png");
        pfTransformImg.scale(pfTransformImg.getWidth() / 2, pfTransformImg.getHeight() / 2);
        
        deathBg = new GreenfootImage("death.png"); 
        
        setImage(idle[0]);
    }

    private GreenfootImage[] loadResized(String prefix, int count) {
        GreenfootImage[] imgs = importSprites(prefix, count);
        for (GreenfootImage img : imgs) img.scale(img.getWidth() / 2, img.getHeight() / 2);
        return imgs;
    }

    @Override
    public void update() {
        if (getWorld() == null) return;
        if (!adjusted) { 
            setLocation(getX(), getY() - 15); 
            adjusted = true; 
        } 
        
        handleNormalCombat();
    }

    public void activatePlantFood() {
        isUsingPF = true; 
        pfStage = -1; 
        frameIndex = 0;
        pfTimer = System.currentTimeMillis();
        pfSoundPlayed = false;
        bgActive = false;
        this.hp = maxHp;
    }

    private void handleNormalCombat() {
        List<Zombie> targets = getObjectsAtOffset(50, 0, Zombie.class); 
        boolean isKO = (punchCount >= 3);

        if (!targets.isEmpty()) {
            playLoop(isKO ? kRight : pRight, 40);
            
            applyDmg(targets, 400, isKO ? 50 : 35, isKO);
        } else {
            playLoop(idle, 40);
            if (System.currentTimeMillis() - lastAttackTime > 1000) punchCount = 0;
        }
    }

    private void removeWithCleanup() {
        if (bgActive && originalWorldBg != null && getWorld() != null) {
            getWorld().setBackground(originalWorldBg);
        }
        if (getWorld() != null) {
            getWorld().removeObject(this);
        }
    }

    private void applyDmg(List<Zombie> targets, int delay, int dmg, boolean ko) {
        if (System.currentTimeMillis() - lastAttackTime > delay) {
            for (Zombie z : targets) {
                if (z.getWorld() != null) z.hit(dmg);
            }
            if (ko) {
                punchCount = 0;
                
            } else {
                punchCount++;
            }
            AudioPlayer.play(80, "bonk.mp3");
            lastAttackTime = System.currentTimeMillis();
        }
    }

    private void playLoop(GreenfootImage[] anim, int delay) {
        if (System.currentTimeMillis() - lastFrameTime > delay) {
            frameIndex = (frameIndex + 1) % anim.length;
            setImage(anim[frameIndex]);
            lastFrameTime = System.currentTimeMillis();
        }
    }

    private void playOnce(GreenfootImage[] anim, int delay) {
        if (System.currentTimeMillis() - lastFrameTime > delay && frameIndex < anim.length - 1) {
            frameIndex++;
            setImage(anim[frameIndex]);
            lastFrameTime = System.currentTimeMillis();
        }
    }

    @Override
    public void hit(int dmg) {
        if (getWorld() == null) return;
        hp -= dmg;
        if (hp <= 0) {
            removeWithCleanup();
        }
    }
}

# BÁO CÁO CẤU TRÚC LỚP BONKCHOY - GAME PROJECT
---

## 1. Giới thiệu chung
Lớp `BonkChoy` kế thừa từ lớp `Plant`, đại diện cho đơn vị thực vật tấn công cận chiến trong trò chơi. Đây là một lớp có logic hoạt ảnh phức tạp nhất, bao gồm nhiều trạng thái (Idle, Attack, Combo, Plant Food) và hệ thống xử lý va chạm phạm vi gần.

---

## 2. Hệ thống tài nguyên và Hoạt ảnh (Animation System)
`BonkChoy` quản lý một thư viện đồ họa đồ sộ để tạo cảm giác mượt mà cho nhân vật:

| Bộ Sprite | Số lượng Frame | Ý nghĩa |
| :--- | :--- | :--- |
| `idle` | 23 | Trạng thái đứng chờ. |
| `pRight` | 10 | Đòn đấm thường bên phải. |
| `kRight` | 15 | Đòn đấm "Knockout" (Combo kết thúc). |
| `pfStart / Loop / End` | 70 | Chuỗi hoạt ảnh khi sử dụng Plant Food. |

> **Kỹ thuật tối ưu:** Tất cả hình ảnh được nạp thông qua hàm bổ trợ `loadResized`, thực hiện việc `scale` giảm kích thước 50% ngay khi khởi tạo để tiết kiệm bộ nhớ GPU và phù hợp với tỷ lệ khung hình của Board.

---

## 3. Logic chiến đấu đặc trưng

### 3.1. Hệ thống Combo (Punch Counter)
Khác với các loại cây bắn đạn từ xa, `BonkChoy` sử dụng biến `punchCount` để theo dõi chuỗi đòn đánh:
* **Đấm thường:** Gây sát thương cơ bản (35 DMG).
* **Đòn kết liễu (KO):** Sau 3 cú đấm thường, nhân vật sẽ tung ra đòn `bonkchoyknockoutone` với sát thương cao hơn (50 DMG) và hoạt ảnh mạnh mẽ hơn.
* **Reset:** Nếu trong 1 giây không có mục tiêu, chuỗi combo sẽ tự động quay về 0.

### 3.2. Kiểm tra phạm vi (Collision Detection)
Sử dụng phương thức `getObjectsAtOffset(50, 0, Zombie.class)` để xác định mục tiêu trong phạm vi cực gần (50 pixels phía trước). Điều này định nghĩa đặc tính "cận chiến" của nhân vật.



---

## 4. Giải thuật xử lý Plant Food (Kỹ năng đặc biệt)
Logic Plant Food của `BonkChoy` được chia làm các giai đoạn (Stages) để xử lý chuỗi hoạt ảnh dài mà không làm treo máy:

1.  **Stage -1 (Start):** Gồng năng lượng và thay đổi hình ảnh biến hình.
2.  **Stage 0 (Loop):** Tung ra hàng loạt cú đấm liên hồi trong một khoảng thời gian xác định.
3.  **Stage 1 (End):** Kết thúc kỹ năng và quay lại trạng thái Idle.

---

## 5. Quản lý bộ nhớ và Hiệu ứng (Cleanup Logic)
Một điểm đặc biệt trong lớp này là hàm `removeWithCleanup()`. Khi `BonkChoy` chết hoặc bị tiêu diệt:
* Hệ thống kiểm tra xem có hiệu ứng nền (`deathBg`) nào đang hoạt động không.
* Nếu có, nó sẽ khôi phục lại Background gốc của `PlayScene` trước khi biến mất. 
* Điều này ngăn chặn lỗi "rác đồ họa" (Visual artifacts) trên màn hình game.



---

## 6. Kết luận
Lớp `BonkChoy` là minh họa điển hình cho việc kết hợp giữa **Logic chiến đấu (Combat Logic)** và **Diễn họa (Animation)**. Việc quản lý tốt các mốc thời gian (`System.currentTimeMillis()`) giúp các đòn đánh của nhân vật có nhịp điệu chính xác, tạo trải nghiệm hành động sống động cho người chơi.

\newpage

In [ ]:
import greenfoot.*;

public class Cactus extends Plant
{
    private GreenfootImage[] idle;
    private GreenfootImage[] shoot;
    private boolean shootOnce = false;
    private boolean shooting = false;
    private long shootDelay = 2400L;
    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;

    public Cactus() {
        maxHp = 100;
        hp = maxHp;
        shoot = importSprites("cactusshoot", 2);
        idle = importSprites("cactus", 4);
    }

    public void hit(int dmg) {
        if (isLiving()) {
            if (!shootOnce) {
                hitFlash(idle, "cactus");
            } else {
                hitFlash(shoot, "cactusshoot");
            }
            hp = hp - dmg;
        }
    }

    public void update() {
        if (getWorld() == null) return;
        
        PlayScene = (PlayScene)getWorld();
        currentFrame = System.nanoTime();

        if (PlayScene.level.zombieRow.get(getYPos()).isEmpty()) {
            shooting = false;
        } else {
            boolean found = false;
            for (Zombie i : PlayScene.level.zombieRow.get(getYPos())) {
                if (i != null && i.getWorld() != null) {
                    if (i.getX() > getX() && i.getX() <= PlayScene.getWidth() + 10) {
                        found = true;
                        break;
                    }
                }
            }
            shooting = found;
        }

        if (!shooting) {
            animate(idle, 150, true);
            lastFrame2 = System.nanoTime();
        } else {
            deltaTime2 = (currentFrame - lastFrame2) / 1000000;
            if (deltaTime2 < shootDelay) {
                animate(idle, 150, true);
                shootOnce = false;
            } else {
                if (!shootOnce) {
                    shootOnce = true;
                    frame = 0;
                }
                if (frame >= 2 && getWorld() != null) {
                    PlayScene.addObject(new Needle(getYPos()), getX() + 30, getY() - 8);
                    lastFrame2 = currentFrame;
                }
                animate(shoot, 150, false);
            }
        }
    }
}

# BÁO CÁO CẤU TRÚC LỚP CACTUS - GAME PROJECT
---

## 1. Giới thiệu chung
Lớp `Cactus` kế thừa từ lớp `Plant`, đại diện cho đơn vị thực vật tấn công tầm xa. Đặc điểm nổi bật của lớp này là khả năng phát hiện kẻ thù trên cùng một hàng và chuyển đổi linh hoạt giữa trạng thái chờ (Idle) và trạng thái tấn công (Shoot).

---

## 2. Quản lý Tài nguyên và Trạng thái
`Cactus` sử dụng hai bộ Sprite riêng biệt để phản ánh hành động của mình:

| Thuộc tính | Loại | Mô tả |
| :--- | :--- | :--- |
| `idle` | `GreenfootImage[]` | Hoạt ảnh 4 khung hình khi không có mục tiêu. |
| `shoot` | `GreenfootImage[]` | Hoạt ảnh 2 khung hình khi thực hiện động tác bắn. |
| `shootDelay` | `long` (2400ms) | Khoảng thời gian giãn cách giữa hai lần bắn liên tiếp. |

---

## 3. Logic Phát hiện Mục tiêu (Target Detection)
Thay vì bắn liên tục, `Cactus` thực hiện kiểm tra thông minh thông qua danh sách Zombie của `WaveManager`:

1.  **Lấy dữ liệu hàng:** Truy cập `PlayScene.level.zombieRow` dựa trên vị trí `getYPos()` của cây.
2.  **Kiểm tra điều kiện:** * Zombie phải tồn tại (`!= null`) và đang ở trong thế giới.
    * Tọa độ X của Zombie phải lớn hơn tọa độ X của cây (nằm phía trước).
    * Zombie phải nằm trong phạm vi màn hình.
3.  **Kết quả:** Nếu thỏa mãn, biến `shooting` được đặt thành `true`.



---

## 4. Cơ chế Hoạt ảnh và Tấn công
Lớp này sử dụng một bộ đếm thời gian riêng (`lastFrame2`, `deltaTime2`) để quản lý nhịp độ bắn độc lập với tốc độ hoạt ảnh:

* **Trạng thái Chờ:** Khi `shooting` là `false`, cây chạy hoạt ảnh `idle` lặp lại.
* **Trạng thái Nạp đạn:** Nếu phát hiện mục tiêu nhưng chưa đủ thời gian `shootDelay`, cây vẫn hiện hoạt ảnh `idle`.
* **Trạng thái Tấn công:**
    * Khi đủ thời gian, cây chuyển sang hoạt ảnh `shoot`.
    * Tại khung hình quy định (`frame >= 2`), một đối tượng đạn mới (`Needle`) sẽ được tạo ra tại tọa độ đầu nòng của cây.



---

## 5. Hiệu ứng Phản hồi (Hit Flash)
Ghi đè (Override) phương thức `hit()` để đảm bảo phản hồi thị giác chính xác:
* Nếu cây đang ở trạng thái chờ, hiệu ứng nháy trắng (`hitFlash`) sẽ áp dụng lên bộ ảnh `idle`.
* Nếu cây đang thực hiện động tác bắn, hiệu ứng sẽ áp dụng lên bộ ảnh `shoot`.
Điều này giúp người chơi luôn nhận thấy cây bị tổn thương dù nó đang thực hiện bất kỳ hành động nào.

---

## 6. Kết luận
Lớp `Cactus` là một ví dụ điển hình về việc tối ưu hóa hiệu suất (chỉ thực hiện logic tấn công khi cần thiết) và quản lý đa trạng thái hoạt ảnh. Việc kết hợp chặt chẽ giữa logic kiểm tra hàng của `WaveManager` và hệ thống `Needle` tạo nên một cơ chế chiến đấu tầm xa hoàn chỉnh.

\newpage

In [ ]:
import greenfoot.*;

public class GatlingPea extends Plant
{
    private GreenfootImage[] idle;
    private GreenfootImage[] shoot;
    private boolean shootOnce = false;
    private int shootCount = 0;
    private boolean resetFrame = false;
    private boolean shooting = false;
    
    private boolean isPoweredUp = false;
    private long powerUpStartTime;
    private final long POWER_UP_DURATION = 3000L;
    private long baseShootDelay = 1100L; 
    private long shootDelay = 1100;
    
    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;

    public GatlingPea() {
        maxHp = 60;
        hp = maxHp;
        shoot = importSprites("GatlingPea", 19);
        idle = importSprites("GatlingPea", 19);
        setImage(idle[0]);
    }

    public void activatePlantFood() {
        this.isPoweredUp = true;
        this.powerUpStartTime = System.currentTimeMillis();
        this.shootDelay = 60L; 
        this.hp = maxHp;
    }

    @Override
    public void hit(int dmg) {
        if (getWorld() != null && isLiving()) {
            hp -= dmg;
        }
    }

    @Override
    public void update() {
        if (getWorld() == null) return;

        if (isPoweredUp) {
            if (System.currentTimeMillis() - powerUpStartTime > POWER_UP_DURATION) {
                isPoweredUp = false;
                shootDelay = baseShootDelay;
            }
        }

        PlayScene = (PlayScene)getWorld();
        currentFrame = System.nanoTime();

        handleAction();
        checkZombieInRow();
    }

    private void handleAction() {
        boolean activeShooting = shooting || isPoweredUp;

        if (!activeShooting) {
            animate(idle, 225, true);
            lastFrame2 = System.nanoTime();
            shootCount = 0;
            resetFrame = false;
        } else {
            deltaTime2 = (currentFrame - lastFrame2) / 1000000;
            
            if (deltaTime2 < shootDelay) {
                if (!isPoweredUp) {
                    animate(idle, 225, true);
                    shootCount = 0;
                    resetFrame = false;
                } else {
                    executeShootingLogic();
                }
            } else {
                if (shootCount >= 5 && !isPoweredUp) {
                    lastFrame2 = currentFrame;
                }
                executeShootingLogic();
            }
        }
    }

    private void executeShootingLogic() {
        if (!resetFrame) {
            setFrame(1);
            resetFrame = true;
        }
        
        if (frame >= 7) {
            int myRow = getYPos();
            if (getWorld() != null && myRow != -1) {
                AudioPlayer.play(80, "throw.mp3", "throw2.mp3");
                PlayScene.addObject(new Pea(myRow), getX() + 25, getY() - 17);
                setFrame(1);
                shootCount++;
            }
        }
        
        int animSpeed = isPoweredUp ? 30 : 70;
        animate(shoot, animSpeed, false);
    }

    private void checkZombieInRow() {
        int myRow = getYPos();
        if (myRow == -1 || PlayScene.level == null) return;

        if (PlayScene.level.zombieRow.get(myRow).isEmpty()) {
            shooting = false;
        } else {
            boolean found = false;
            for (Zombie i : PlayScene.level.zombieRow.get(myRow)) {
                if (i.getWorld() != null && i.getX() > getX() && i.getX() <= PlayScene.getWidth() + 10) {
                    found = true;
                    break;
                }
            }
            shooting = found;
        }
    }
}

# BÁO CÁO CẤU TRÚC LỚP GATLINGPEA - GAME PROJECT
---

## 1. Giới thiệu chung
Lớp `GatlingPea` kế thừa từ lớp `Plant`, là đơn vị tấn công tầm xa hạng nặng. Khác với các đơn vị bắn đậu thông thường, GatlingPea có khả năng bắn liên tiếp 5 viên đậu trong một chu kỳ tấn công và có chế độ bắn siêu tốc khi được kích hoạt kỹ năng đặc biệt.

---

## 2. Các thuộc tính và Trạng thái chiến đấu
Cấu trúc lớp quản lý chặt chẽ giữa hai trạng thái: bắn thường và bắn khi có Plant Food.

| Thuộc tính | Kiểu dữ liệu | Ý nghĩa kỹ thuật |
| :--- | :--- | :--- |
| `shootCount` | `int` | Đếm số lượng đạn đã bắn trong một đợt (tối đa 5 viên). |
| `isPoweredUp` | `boolean` | Trạng thái kích hoạt kỹ năng đặc biệt (Plant Food). |
| `baseShootDelay` | `long` | Thời gian chờ giữa các đợt bắn thường (1100ms). |
| `shootDelay` | `long` | Tốc độ bắn thực tế, thay đổi dựa trên trạng thái Power-up. |

---

## 3. Logic Tấn công Liên thanh (Burst Fire Logic)
Điểm đặc sắc của GatlingPea nằm ở hàm `executeShootingLogic()`. Thuật toán được thiết kế để bắn ra một chuỗi đạn thay vì một viên duy nhất:

1.  **Reset khung hình:** Sử dụng biến `resetFrame` để đưa hoạt ảnh về frame đầu tiên khi bắt đầu đợt bắn mới.
2.  **Kích hoạt bắn:** Khi hoạt ảnh đạt đến khung hình số 7 (`frame >= 7`), một đối tượng `Pea` được khởi tạo.
3.  **Vòng lặp bắn:** Thay vì chờ đợi delay dài, nó sử dụng `setFrame(1)` để lặp lại nhanh hoạt ảnh bắn cho đến khi `shootCount` đạt giới hạn.



---

## 4. Cơ chế Plant Food (Power-up Mode)
Khi hàm `activatePlantFood()` được gọi, GatlingPea chuyển sang trạng thái "quá tải":

* **Tăng tốc độ cực đại:** `shootDelay` giảm từ 1100ms xuống còn **60ms**.
* **Tăng tốc hoạt ảnh:** Tốc độ diễn họa (`animSpeed`) chuyển từ 70ms xuống còn **30ms** mỗi frame.
* **Thời gian tác dụng:** Duy trì trong 3000ms (3 giây) nhờ biến `POWER_UP_DURATION`.



---

## 5. Thuật toán Kiểm tra Mục tiêu và Điều phối Action
Hàm `handleAction()` đóng vai trò điều phối giữa việc hiển thị hoạt ảnh chờ (`idle`) và thực thi logic bắn (`executeShootingLogic`):

* Nếu **không có Zombie** hoặc **đang chờ nạp đạn:** Chạy hoạt ảnh `idle` chậm (225ms/frame).
* Nếu **đang Power-up:** Bỏ qua mọi thời gian chờ, ép nhân vật vào trạng thái bắn liên tục.
* Nếu **đang bắn thường:** Sau khi bắn đủ 5 viên (`shootCount >= 5`), reset bộ đếm thời gian `lastFrame2` để tạo khoảng nghỉ.

---

## 6. Kết luận
Lớp `GatlingPea` thể hiện mức độ phức tạp cao trong việc quản lý logic thời gian (`System.nanoTime()` và `System.currentTimeMillis()`). Việc kết hợp giữa số lượng đạn mỗi đợt, tốc độ hoạt ảnh biến thiên và khả năng tương tác với hệ thống Plant Food làm cho GatlingPea trở thành một trong những đối tượng có hỏa lực mạnh nhất và logic vận hành tinh vi nhất trong trò chơi.

\newpage

In [ ]:
import greenfoot.*;

public class MegaGatlingPea extends Plant
{
    private GreenfootImage[] idle;
    private GreenfootImage[] shoot;
    private boolean shootOnce = false;
    private int shootCount = 0;
    private boolean resetFrame = false;
    private boolean shooting = false;
    
    private boolean isPoweredUp = false;
    private long powerUpStartTime;
    private final long POWER_UP_DURATION = 3000L;
    private long baseShootDelay = 600L; 
    private long shootDelay = 600L;
    
    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;

    public MegaGatlingPea() {
        maxHp = 60;
        hp = maxHp;
      
        shoot = scaleImages(importSprites("MegaGatlingPea_shoot", 31), 0.75);
        idle = scaleImages(importSprites("MegaGatlingPea_idle", 31), 0.75);
        setImage(idle[0]);
    }

    @Override
    public void addedToWorld(World world) {
        super.addedToWorld(world); 
        setLocation(getX() + 15, getY() - 15);
    }

    private GreenfootImage[] scaleImages(GreenfootImage[] images, double factor) {
        for (GreenfootImage img : images) {
            int newWidth = (int)(img.getWidth() * factor);
            int newHeight = (int)(img.getHeight() * factor);
            img.scale(newWidth, newHeight);
        }
        return images;
    }

    public void activatePlantFood() {
        this.isPoweredUp = true;
        this.powerUpStartTime = System.currentTimeMillis();
        this.shootDelay = 60L; 
        this.hp = maxHp;
    }

    @Override
    public void hit(int dmg) {
        if (getWorld() != null && isLiving()) {
            hp -= dmg;
        }
    }

    @Override
    public void update() {
        if (getWorld() == null) return;

        if (isPoweredUp) {
            if (System.currentTimeMillis() - powerUpStartTime > POWER_UP_DURATION) {
                isPoweredUp = false;
                shootDelay = baseShootDelay;
            }
        }

        currentFrame = System.nanoTime();

        handleAction();
        checkZombieInRow();
    }

    private void handleAction() {
        boolean activeShooting = shooting || isPoweredUp;

        if (!activeShooting) {
            animate(idle, 225, true);
            lastFrame2 = System.nanoTime();
            shootCount = 0;
            resetFrame = false;
        } else {
            deltaTime2 = (currentFrame - lastFrame2) / 1000000;
            
            if (deltaTime2 < shootDelay) {
                if (!isPoweredUp) {
                    animate(idle, 225, true);
                    shootCount = 0;
                    resetFrame = false;
                } else {
                    executeShootingLogic();
                }
            } else {
                if (shootCount >= 7 && !isPoweredUp) {
                    lastFrame2 = currentFrame;
                }
                executeShootingLogic();
            }
        }
    }

    private void executeShootingLogic() {
        if (!resetFrame) {
            setFrame(1);
            resetFrame = true;
        }
        
        if (frame >= 12) {
            int myRow = getYPos();
            if (getWorld() != null && myRow != -1) {
                AudioPlayer.play(80, "throw.mp3", "throw2.mp3");
                PlayScene.addObject(new FirePea(myRow), getX() + 20, getY());
                setFrame(1);
                shootCount++;
            }
        }
        
        int animSpeed = isPoweredUp ? 20 : 40;
        animate(shoot, animSpeed, false);
    }

    private void checkZombieInRow() {
        int myRow = getYPos();
        if (myRow == -1 || PlayScene.level == null) return;

        if (PlayScene.level.zombieRow.get(myRow).isEmpty()) {
            shooting = false;
        } else {
            boolean found = false;
            for (Zombie i : PlayScene.level.zombieRow.get(myRow)) {
                if (i.getWorld() != null && i.getX() > getX() && i.getX() <= PlayScene.getWidth() + 10) {
                    found = true;
                    break;
                }
            }
            shooting = found;
        }
    }
}

# BÁO CÁO CẤU TRÚC LỚP MEGAGATLINGPEA - GAME PROJECT
---

## 1. Giới thiệu chung
`MegaGatlingPea` là lớp thực vật tấn công cấp cao nhất trong trò chơi, được phát triển dựa trên nền tảng của `GatlingPea`. Lớp này không chỉ kế thừa các đặc tính bắn liên thanh mà còn được nâng cấp về sát thương (bắn đạn lửa) và mật độ hỏa lực, đóng vai trò là đơn vị phòng thủ chủ lực.

---

## 2. Đặc điểm kỹ thuật và Đồ họa
Để tương xứng với sức mạnh, `MegaGatlingPea` có bộ tài nguyên hình ảnh chi tiết và cơ chế căn chỉnh vị trí riêng biệt:

| Thông số | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **Số khung hình** | 31 Frames | Tạo chuyển động mượt mà cho cả trạng thái Idle và Shoot. |
| **Tỷ lệ Scale** | 0.75 | Sử dụng hàm `scaleImages` để tối ưu kích thước hiển thị nhưng vẫn giữ độ nét cao. |
| **Offset Vị trí** | X+15, Y-15 | Ghi đè `addedToWorld` để tự động điều chỉnh tọa độ, giúp nhân vật nằm khớp với ô lưới. |

---

## 3. Logic Hỏa lực Tối thượng (Mega Burst Logic)
Logic bắn của MegaGatlingPea được thiết kế để gây áp lực lớn lên hàng hàng phòng thủ của Zombie:

* **Sát thương hệ Lửa:** Thay vì bắn đạn thường, hàm `executeShootingLogic()` tạo ra đối tượng `FirePea`, nhân đôi sức mạnh tấn công.
* **Mật độ đạn:** Số lượng đạn trong một chu kỳ bắn thường được nâng lên **7 viên** (`shootCount >= 7`), so với 5 viên của GatlingPea thông thường.
* **Điểm kích hoạt (Trigger Point):** Đạn được tạo ra tại khung hình thứ 12 (`frame >= 12`), đảm bảo khớp với hành động "nhả đạn" trong hoạt ảnh.



---

## 4. Chế độ Siêu cấp (Plant Food Overload)
Khi kích hoạt `activatePlantFood()`, MegaGatlingPea đạt đến trạng thái hỏa lực không giới hạn:

1. **Tốc độ bắn thần tốc:** `shootDelay` giảm xuống mức tối thiểu **60ms**.
2. **Tốc độ diễn họa:** `animSpeed` đạt **20ms/frame**, tạo hiệu ứng rung động cực nhanh đặc trưng của vũ khí Gatling.
3. **Thời gian hiệu lực:** Duy trì hỏa lực rực lửa trong 3 giây trước khi quay lại trạng thái cơ bản.



---

## 5. Cải tiến thuật toán Điều khiển
Lớp này sử dụng một hệ thống quản lý trạng thái kép cực kỳ chặt chẽ:
* **ActiveShooting:** Biến logic kết hợp giữa việc phát hiện Zombie (`shooting`) và trạng thái thức ăn thực vật (`isPoweredUp`). 
* **Tự động quét hàng:** Hàm `checkZombieInRow` liên tục cập nhật trạng thái mục tiêu, giúp cây ngừng bắn ngay lập tức khi hàng trống để tiết kiệm tài nguyên hệ thống.

---

## 6. Kết luận
`MegaGatlingPea` là minh chứng cho việc tối ưu hóa và tái sử dụng mã nguồn. Bằng cách tinh chỉnh các thông số thời gian (`600L` base delay) và thay đổi loại đối tượng tương tác (`FirePea`), lớp này đã tạo ra một thực thể có sức mạnh vượt trội, mang lại chiều sâu chiến thuật và trải nghiệm hỏa lực mãn nhãn cho người chơi.

\newpage

In [ ]:
import greenfoot.*;

public class Peashooter extends Plant
{
    private GreenfootImage[] idle;
    private GreenfootImage[] shoot;
    private boolean shootOnce = false;
    private boolean shooting = false;
   
    private boolean isPoweredUp = false;
    private long powerUpStartTime;
    private final long POWER_UP_DURATION = 3000L;
    private long baseShootDelay = 1500L; 
    private long shootDelay = 1500L; 

    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;
    
    public Peashooter() {
        maxHp = 60;
        hp = maxHp;
        shoot = importSprites("peashootershoot", 3);
        idle = importSprites("peashooter", 9);
        setImage(idle[0]);
    }
   
    @Override
    public void hit(int dmg) {
        if (getWorld() != null && isLiving()) {
            if (!shootOnce) hitFlash(idle, "peashooter");
            else hitFlash(shoot, "peashootershoot");  
            hp -= dmg;
        }
    }

    
    public void activatePlantFood() {
        this.isPoweredUp = true;
        this.powerUpStartTime = System.currentTimeMillis();
        this.shootDelay = 300L; 
        
    }

    @Override
    public void update() {
        if (getWorld() == null) return;
        
        
        if (isPoweredUp) {
            if (System.currentTimeMillis() - powerUpStartTime > POWER_UP_DURATION) {
                isPoweredUp = false;
                shootDelay = baseShootDelay;
            }
        }

        PlayScene = (PlayScene)getWorld();
        currentFrame = System.nanoTime();

        handleAnimationAndShooting();
        
        if (getWorld() == null) return;
        checkZombieInRow();
    }

    private void handleAnimationAndShooting() {
        
        boolean activeShooting = shooting || isPoweredUp;

        if (!activeShooting) {
            animate(idle, 300, true);
            lastFrame2 = System.nanoTime();
        } else {
            deltaTime2 = (currentFrame - lastFrame2) / 1000000;
            if (deltaTime2 < shootDelay) {
               
                if (!isPoweredUp) {
                    animate(idle, 1, true); 
                } else {
                    
                    animate(shoot, 1, false); 
                }
                shootOnce = false;
            } else {
                if (!shootOnce) {
                    shootOnce = true;
                    frame = 0; 
                }
                
                if (frame >= 1 && shootOnce) {
                    int myRow = getYPos();
                    if (getWorld() != null && myRow != -1) {
                        AudioPlayer.play(80, "throw.mp3", "throw2.mp3");
                        PlayScene.addObject(new Pea(myRow), getX() + 25, getY() - 17);
                        lastFrame2 = currentFrame;
                        shootOnce = false; 
                    }
                }
                
                int animDelay = isPoweredUp ? 2 : 10;
                animate(shoot, animDelay, false);
            }
        }
    }

    private void checkZombieInRow() {
        int myRow = getYPos(); 
        if (myRow == -1 || PlayScene.level == null) return;

        if (PlayScene.level.zombieRow.get(myRow).isEmpty()) {
            shooting = false;
        } else {
            boolean found = false;
            for (Zombie i : PlayScene.level.zombieRow.get(myRow)) {
                if (i.getWorld() != null && i.getX() > getX() && i.getX() <= PlayScene.getWidth() + 10) {
                    found = true;
                    break;
                }
            }
            shooting = found;
        }
    }
}

# BÁO CÁO CẤU TRÚC LỚP PEASHOOTER - GAME PROJECT
---

## 1. Giới thiệu chung
`Peashooter` là đơn vị tấn công tầm xa cơ bản nhất trong trò chơi. Lớp này kế thừa từ lớp `Plant`, đóng vai trò là "mẫu chuẩn" cho các đơn vị bắn đạn khác. Nó thực hiện logic kiểm tra mục tiêu theo hàng và điều khiển hoạt ảnh bắn đạn dựa trên trạng thái chiến đấu thực tế.

---

## 2. Quản lý Tài nguyên và Thông số
Lớp `Peashooter` định nghĩa các thông số sinh tồn và tấn công cơ bản:

| Thuộc tính | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **maxHp** | 60 | Lượng máu cơ bản của đơn vị. |
| **idle** | 9 Frames | Hoạt ảnh trạng thái đứng chờ. |
| **shoot** | 3 Frames | Hoạt ảnh trạng thái bắn đạn. |
| **baseShootDelay** | 1500ms | Khoảng cách giữa hai lần bắn ở trạng thái thường. |

---

## 3. Cơ chế Kiểm tra Mục tiêu (Row Scanning)
Hàm `checkZombieInRow()` thực hiện quét hàng mỗi khung hình để tối ưu hóa hành động:
* **Truy vấn dữ liệu:** Kiểm tra danh sách `zombieRow` của hàng hiện tại từ `PlayScene.level`.
* **Xác định vị trí:** Chỉ kích hoạt trạng thái `shooting` nếu Zombie nằm phía trước cây (`i.getX() > getX()`) và vẫn còn trong phạm vi bản đồ.
* **Tối ưu:** Nếu hàng trống, cây sẽ lập tức quay lại trạng thái `idle` để giảm tải tính toán logic bắn.



---

## 4. Logic Tấn công và Plant Food
`Peashooter` sử dụng một hệ thống quản lý thời gian kép để điều khiển nhịp độ bắn:

### 4.1. Trạng thái thường
Khi `shooting = true`, cây sẽ đợi đủ `shootDelay` (1.5 giây). Khi đạt mốc:
1. Đặt `frame = 0` để bắt đầu hoạt ảnh bắn.
2. Tại khung hình chỉ định (`frame >= 1`), một đối tượng `Pea` (Đậu) sẽ được thêm vào thế giới.
3. Phát âm thanh ngẫu nhiên giữa `throw.mp3` và `throw2.mp3`.

### 4.2. Chế độ Plant Food (Powered Up)
Khi kích hoạt `activatePlantFood()`:
* **Tăng tốc độ bắn:** `shootDelay` giảm từ 1500ms xuống còn **300ms**.
* **Tăng tốc hoạt ảnh:** `animDelay` giảm từ 10 xuống **2**, giúp hoạt ảnh bắn diễn ra cực nhanh.
* **Thời gian tác dụng:** Hiệu ứng duy trì trong 3 giây (`POWER_UP_DURATION`).



---

## 5. Hiệu ứng Phản hồi Thống nhất
Lớp ghi đè phương thức `hit()` để tích hợp hệ thống `hitFlash` từ lớp cha:
* Hệ thống tự động nhận diện cây đang ở bộ ảnh `idle` hay `shoot` để thực hiện hiệu ứng nháy trắng chính xác.
* Điều này đảm bảo tính nhất quán về mặt thị giác (Visual Consistency), giúp người chơi nhận biết phản hồi va chạm ngay lập tức.

---

## 6. Kết luận
Lớp `Peashooter` là minh chứng rõ nhất cho sức mạnh của tính kế thừa. Bằng cách sử dụng các phương thức có sẵn từ `Plant` và `SpriteAnimator`, nó tập trung hoàn toàn vào việc điều khiển nhịp độ bắn đạn. Đây là nền tảng quan trọng để phát triển các biến thể phức tạp hơn như `GatlingPea` hay `MegaGatlingPea`.

\newpage

In [ ]:
import greenfoot.*; 
public class PotatoMine extends Plant
{
    private GreenfootImage[] idle;
    private GreenfootImage[] arm;
    private boolean playOnce = false;
    public boolean armed = false;
    private long lastFrame2 = System.nanoTime();
    private boolean playSFX = false;
    private long deltaTime2 = System.nanoTime();
    public PotatoMine() {
        idle = importSprites("potato", 5);
        arm = importSprites("potatomine", 3);
        maxHp = 60;
        hp = maxHp;
        currentFrame = System.nanoTime();
        lastFrame2 = System.nanoTime();
    }
    
    public void update() {
        currentFrame = System.nanoTime();
        deltaTime2 = (currentFrame - lastFrame2) / 1000000;
        if (deltaTime2 > 22000L) {
            armed = true;
            if (!playOnce) {
                if (!playSFX) {
                    playSFX = true;
                    AudioPlayer.play(70, "dirt_rise.mp3");
                }
                animate(arm, 200, false);
                if (frame > 2) {
                    playOnce = true;
                }
            } else {
                animate(idle, 200, true);
            }
            checkExplosion();
            
        }
        

    }
    public void hit(int dmg) {
        if (isLiving() && !armed) {
            
            
            hp = hp-dmg;
        }
    }
    public void checkExplosion() {
        if (PlayScene.level.zombieRow.get(getYPos()).size() == 0) {
            
        } else {
            
            for (Zombie i : PlayScene.level.zombieRow.get(getYPos())) {
                if (Math.abs(i.getX() - getX()) < 28) {
                    
                    getWorld().addObject(new Explosion(PlayScene.level.zombieRow.get(getYPos())), getX(), getY()-25);
                    ((PlayScene)getWorld()).board.removePlant(getXPos(), getYPos());
                    return;
                } 
            }
                                    
        }
    }
    
  
}


# BÁO CÁO CẤU TRÚC LỚP POTATOMINE - GAME PROJECT
---

## 1. Giới thiệu chung
`PotatoMine` là một loại thực vật đặc biệt thuộc nhóm phòng thủ bị động. Cơ chế hoạt động của nó chia làm hai giai đoạn: giai đoạn chuẩn bị (vùi mình dưới đất) và giai đoạn kích hoạt (sẵn sàng nổ). Đây là lớp minh họa rõ nét cho việc quản lý trạng thái dựa trên thời gian thực.

---

## 2. Quản lý Trạng thái và Tài nguyên
Lớp sử dụng hai bộ Sprite khác nhau để thể hiện mức độ sẵn sàng:

| Bộ Sprite | Số lượng Frame | Ý nghĩa thị giác |
| :--- | :--- | :--- |
| `arm` | 3 | Hoạt ảnh Potato Mine đang trồi lên khỏi mặt đất. |
| `idle` | 5 | Hoạt ảnh khi Potato Mine đã sẵn sàng kích nổ (đầu nhú lên). |

* **Thông số đặc trưng:** Biến `armed` (boolean) dùng để kiểm soát việc Potato Mine đã có thể gây sát thương hay chưa.

---

## 3. Logic Kích hoạt theo Thời gian (Arming Delay)
Potato Mine không có tác dụng ngay khi vừa trồng. Thuật toán sử dụng `deltaTime2` để tạo ra một khoảng thời gian chờ:

* **Thời gian chờ:** 22,000 mili giây (22 giây).
* **Quy trình:** 1. Khi `deltaTime2 > 22000L`, biến `armed` chuyển sang `true`.
    2. Chạy hoạt ảnh `arm` một lần duy nhất (`playOnce`).
    3. Phát âm thanh hiệu ứng `dirt_rise.mp3` để thông báo cho người chơi.
    4. Chuyển sang trạng thái `idle` lặp lại.



---

## 4. Cơ chế Kích nổ và Va chạm
Hàm `checkExplosion()` chỉ hoạt động khi Potato Mine đã ở trạng thái `armed`. Logic xử lý va chạm như sau:

1. **Kiểm tra hàng:** Truy cập `zombieRow` của hàng hiện tại (thông qua `getYPos()`).
2. **Tính toán khoảng cách:** Sử dụng hàm trị tuyệt đối `Math.abs(i.getX() - getX())`.
3. **Điều kiện nổ:** Nếu khoảng cách giữa Potato Mine và Zombie nhỏ hơn **28 pixels**:
    * Tạo đối tượng `Explosion` (Vụ nổ) tại vị trí hiện tại.
    * Gọi lệnh `board.removePlant()` để xóa dữ liệu mìn khỏi bàn cờ.
    * Hủy đối tượng ngay lập tức.

---

## 5. Đặc tính Sinh tồn (Hit Logic)
Lớp `PotatoMine` có một quy tắc sinh tồn rất đặc biệt trong hàm `hit()`:
* **Giai đoạn chưa kích hoạt:** Zombie có thể ăn (gây sát thương) và tiêu diệt Potato Mine.
* **Giai đoạn đã kích hoạt (Armed):** Potato Mine trở nên "bất tử" trước các đòn tấn công cắn của Zombie vì lúc này logic va chạm nổ sẽ ưu tiên chạy trước, Zombie chạm vào là nổ ngay lập tức thay vì kịp gây sát thương.



---

## 6. Kết luận
Lớp `PotatoMine` là một sự kết hợp khéo léo giữa quản lý thời gian (`System.nanoTime`) và xử lý va chạm phạm vi gần. Việc thiết kế độ trễ khi kích hoạt không chỉ mang lại tính cân bằng cho trò chơi mà còn đòi hỏi người chơi phải có chiến thuật đặt mìn từ sớm.

\newpage

In [ ]:
import greenfoot.*; 
public class Repeater extends Plant
{
    private GreenfootImage[] idle;
    private GreenfootImage[] shoot;
    private boolean shootOnce = false;
    private int shootCount = 0;
    private boolean resetFrame = false;
    private boolean shooting = false;
    private long shootDelay = 1700L;
    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;
    
    public Repeater() {
        maxHp = 60;
        hp = maxHp;
        shoot = importSprites("repeatershoot", 3);
        idle = importSprites("repeater", 7);
    }
   
    public void hit(int dmg) {
        if (isLiving()) {
            if (!shootOnce) {
                hitFlash(idle, "repeater");
            } else {
                hitFlash(shoot, "repeatershoot");  
                
            }
            hp = hp-dmg;
        }
    }
    public void update() {
        if (getWorld() == null) return;
        
        PlayScene world = (PlayScene)getWorld();
        currentFrame = System.nanoTime();
        if (!shooting) {
            animate(idle, 225, true);
            lastFrame2 = System.nanoTime();
        } else {
            
            deltaTime2 = (currentFrame - lastFrame2) / 1000000;
            if (deltaTime2 < shootDelay) {
                animate(idle, 225, true);
                shootCount = 0;
                resetFrame = false;
            } else {
                if (shootCount >= 3) {
                    lastFrame2 = currentFrame;
                }
                if (!resetFrame) {
                    setFrame(1);
                    resetFrame = true;
                }
                
                if (frame >= 3) {
                    AudioPlayer.play(80, "throw.mp3", "throw2.mp3");
                    
                    if (getWorld() != null) {
                        world.addObject(new Pea(getYPos()), getX() + 25, getY() - 17);
                    }
                    
                    PlayScene.addObject(new Pea(getYPos()), getX()+25,getY()-17);
                    setFrame(1);
                    setImage("repeatershoot1.png");
                    shootCount++;
                }
                animate(shoot, 70, false);
                
                
            }
            
            
        }
        if (world.level.zombieRow.get(getYPos()).size() == 0) {
            shooting = false;
        } else {
            boolean foundZombie = false;
            for (Zombie i : world.level.zombieRow.get(getYPos())) {
                
                if (i != null && i.getWorld() != null && i.getX() > getX() && i.getX() <= world.getWidth() + 10) {
                    foundZombie = true;
                    break;
                }
            }
            shooting = foundZombie;                      
        }
    }
 
}


# BÁO CÁO CẤU TRÚC LỚP REPEATER - GAME PROJECT
---

## 1. Giới thiệu chung
`Repeater` là một đơn vị tấn công tầm xa trung cấp, được nâng cấp trực tiếp từ `Peashooter`. Đặc điểm nhận dạng cốt lõi của lớp này là khả năng bắn ra hai viên đậu (`Pea`) trong mỗi đợt tấn công, giúp tăng gấp đôi hỏa lực trên cùng một hàng phòng thủ.

---

## 2. Quản lý Tài nguyên Hoạt ảnh
`Repeater` quản lý hai trạng thái diễn họa chính để phản hồi hành động của cây:

| Bộ Sprite | Số lượng Frame | Mô tả |
| :--- | :--- | :--- |
| `idle` | 7 | Hoạt ảnh trạng thái chờ khi không có mục tiêu. |
| `shoot` | 3 | Hoạt ảnh thực hiện động tác nhả đạn. |

* **Độ trễ tấn công (`shootDelay`):** 1700 mili giây (1.7 giây) cho mỗi chu kỳ bắn hai viên.

---

## 3. Logic "Bắn Kép" (Double Shooting Logic)
Đây là phần quan trọng nhất trong mã nguồn của `Repeater`. Thay vì chỉ bắn một lần rồi chờ, nó sử dụng biến đếm `shootCount` để kiểm soát chuỗi đạn:

1.  **Kích hoạt đợt bắn:** Khi phát hiện Zombie, hệ thống bắt đầu tính `deltaTime2`. 
2.  **Vòng lặp trong đợt:**
    * Khi hoạt ảnh đạt khung hình cuối (`frame >= 3`), một viên đậu được thêm vào.
    * Ngay lập tức, mã nguồn gọi `setFrame(1)` để **ép hoạt ảnh quay lại từ đầu**, bắt đầu lần bắn thứ hai.
    * Biến `shootCount` tăng lên sau mỗi lần nhả đạn.
3.  **Kết thúc đợt:** Khi `shootCount >= 3` (thực tế bắn 2-3 viên tùy cấu hình), bộ đếm thời gian `lastFrame2` mới được reset để bắt đầu khoảng nghỉ `shootDelay`.



---

## 4. Quét mục tiêu và Phản hồi Va chạm
Tương tự các đơn vị bắn đậu khác, `Repeater` sử dụng cơ chế quét hàng thông minh:

* **Phát hiện Zombie:** Duyệt danh sách `zombieRow` của hàng hiện tại. Chỉ kích hoạt `shooting = true` nếu Zombie nằm phía trước cây và còn trong màn hình.
* **Hit Flash:** Phương thức `hit()` được tối ưu để hiệu ứng nháy trắng xuất hiện đúng trên bộ ảnh đang hiển thị (Dù cây đang đứng yên hay đang bắn).

---

## 5. Giải thuật xử lý khung hình (Frame Manipulation)
Một kỹ thuật đáng chú ý trong lớp này là việc ép hình ảnh trực tiếp:
* `setImage("repeatershoot1.png")`: Sau khi bắn, cây chủ động đặt lại hình ảnh để tạo cảm giác giật (recoil).
* Biến `resetFrame` giúp đồng bộ hóa việc bắt đầu hoạt ảnh bắn ngay lập tức khi đủ thời gian chờ, tránh việc bị khựng do lệch khung hình cũ.



---

## 6. Kết luận
Lớp `Repeater` là một bước tiến về logic so với `Peashooter`. Việc sử dụng các biến kiểm soát trạng thái như `shootCount` và `resetFrame` cho phép thực hiện các hành động phức tạp hơn trong một chu kỳ `act()`. Đây là nền tảng quan trọng để phát triển các đơn vị bắn liên thanh cao cấp hơn như `GatlingPea`.

\newpage

In [ ]:
import greenfoot.*; 
public class Sunflower extends Plant
{
    private int sunProductionTime = 10000;
    private GreenfootImage[] idle;
    private boolean test = false;
    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;
    public Sunflower() {
        idle = importSprites("sunfloweridle", 8);
        maxHp = 60;
        hp = maxHp;
    }
    public void update(){
        produceSun();
        animate(idle, 200, true);

    }
    public void hit(int dmg) {
        if (isLiving()) {
            hitFlash(idle, "sunfloweridle");
            
            hp = hp-dmg;
        }
    }
    public void produceSun() {
    
    deltaTime2 = (System.nanoTime() - lastFrame2) / 1000000;
    
    
    if (deltaTime2 > sunProductionTime) {
        lastFrame2 = System.nanoTime(); 
        
        hitFlash(idle, "sunfloweridle");
        test = true;
        
        
        PlayScene.addObject(new Sun(5), getX(), getY() - 10);
    }
}
  
}


# BÁO CÁO CẤU TRÚC LỚP SUNFLOWER - GAME PROJECT
---

## 1. Giới thiệu chung
`Sunflower` là đơn vị sản xuất tài nguyên chính trong trò chơi. Lớp này kế thừa từ `Plant`, không có khả năng tấn công nhưng đóng vai trò sống còn trong việc cung cấp năng lượng mặt trời (`Sun`) để người chơi có thể trồng các loại thực vật khác.

---

## 2. Thông số kỹ thuật và Hoạt ảnh
Hướng dương được thiết kế để hoạt động liên tục và ổn định:

| Thuộc tính | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **maxHp** | 60 | Lượng máu cơ bản. |
| **idle** | 8 Frames | Hoạt ảnh trạng thái lắc lư nhẹ nhàng khi sản xuất năng lượng. |
| **sunProductionTime** | 10,000 ms | Chu kỳ sản xuất mặt trời (10 giây một lần). |

---

## 3. Logic Sản xuất Tài nguyên (Sun Generation)
Hàm `produceSun()` quản lý việc tạo ra năng lượng dựa trên thời gian thực (`System.nanoTime()`):

1. **Tính toán thời gian:** Sử dụng biến `deltaTime2` để đo khoảng cách mili giây từ lần tạo mặt trời gần nhất.
2. **Kích hoạt sự kiện:** Khi `deltaTime2 > 10000`:
    * **Hiệu ứng phản hồi:** Gọi `hitFlash` để làm cây nháy sáng lên, tạo tín hiệu thị giác báo hiệu sắp có mặt trời.
    * **Tạo đối tượng:** Một đối tượng `Sun` với giá trị tài nguyên (ví dụ: 5 hoặc 25 tùy cấu hình) sẽ được thêm vào thế giới tại tọa độ của cây.
    * **Reset:** Đặt lại mốc `lastFrame2` để bắt đầu chu kỳ 10 giây tiếp theo.



---

## 4. Đặc điểm Hoạt ảnh và Tương tác
* **Hoạt ảnh liên tục:** Trong hàm `update()`, lệnh `animate(idle, 200, true)` luôn được gọi để đảm bảo hoa hướng dương luôn chuyển động, tạo sự sống động cho bàn cờ.
* **Phản hồi va chạm:** Phương thức `hit()` tích hợp `hitFlash` để người chơi nhận biết khi hoa hướng dương đang bị Zombie tấn công, giúp kịp thời đưa ra phương án bảo vệ.

---

## 5. Vai trò chiến thuật trong Hệ thống
Trong cấu trúc game, `Sunflower` là đơn vị trung tâm điều phối nhịp độ:
* **Input:** Thời gian thực trôi qua.
* **Output:** Đối tượng `Sun` chuyển vào tổng quỹ tài nguyên của người chơi.
* **Sự phụ thuộc:** Nếu `Sunflower` bị tiêu diệt, chuỗi cung ứng năng lượng sẽ bị gián đoạn, trực tiếp dẫn đến thất bại.



---

## 6. Kết luận
Lớp `Sunflower` minh họa cho một dạng logic "Passive" (Bị động) nhưng cực kỳ quan trọng. Việc sử dụng thành thạo biến thời gian để điều khiển sự kiện tạo vật thể (`addObject`) là nền tảng để xây dựng các cơ chế hỗ trợ khác trong trò chơi.

\newpage

In [ ]:
import greenfoot.*; 
public class Torchwood extends Plant
{
    private GreenfootImage[] idle;
    
    public Torchwood() {
        idle = importSprites("firewood", 5);
        maxHp = 100; 
        hp = maxHp;
    }

    public void update() {
        animate(idle, 200, true);
        checkAndUpgradePea(); 
    }

        private void checkAndUpgradePea() {
        
        Pea p = (Pea) getOneIntersectingObject(Pea.class);
        
        if (p != null && p.getWorld() != null) {
            int curX = p.getX();
            int curY = p.getY();
            int rowIdx = p.getYPos(); 
            
            getWorld().removeObject(p);
            
            
            FirePea fp = new FirePea(rowIdx); 
            getWorld().addObject(fp, curX, curY);
        }
    }

    public void hit(int dmg) {
        if (isLiving()) {
            
            hitFlash(idle, "firewood");
            hp = hp - dmg;
            
            if (hp <= 0) {
                getWorld().removeObject(this);
            }
        }
    }
}

# BÁO CÁO CẤU TRÚC LỚP TORCHWOOD - GAME PROJECT
---

## 1. Giới thiệu chung
`Torchwood` là một đơn vị thực vật hỗ trợ (Support Unit). Lớp này không sở hữu khả năng tấn công trực tiếp nhưng đóng vai trò là một "bộ lọc năng lượng", giúp nâng cấp sức mạnh cho các loại đạn đậu đi ngang qua nó. Đây là một mắt xích quan trọng trong việc tối ưu hóa sát thương hàng phòng thủ.

---

## 2. Thông số kỹ thuật và Hoạt ảnh
`Torchwood` duy trì một trạng thái rực lửa liên tục để sẵn sàng tương tác với đạn:

| Thuộc tính | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **maxHp** | 100 | Lượng máu cao hơn mức trung bình để trụ vững ở hàng trước. |
| **idle** | 5 Frames | Hoạt ảnh ngọn lửa bùng cháy liên tục (`firewood`). |
| **Hàm update** | `animate` | Luôn chạy hoạt ảnh lặp lại để tạo hiệu ứng thị giác về nhiệt độ. |

---

## 3. Logic Biến đổi Đối tượng (Projectile Upgrading)
Trái tim của lớp này nằm ở phương thức riêng tư `checkAndUpgradePea()`. Thuật toán thực hiện việc thay thế đối tượng theo thời gian thực:

1. **Phát hiện va chạm:** Sử dụng `getOneIntersectingObject(Pea.class)` để kiểm tra xem có viên đậu thường nào đang chạm vào `Torchwood` hay không.
2. **Thu thập dữ liệu:** Nếu tìm thấy đạn `Pea`, hệ thống sẽ ghi nhớ tọa độ (`curX`, `curY`) và chỉ số hàng (`rowIdx`) của viên đạn đó.
3. **Thay thế đối tượng:**
    * Xóa viên đậu thường (`Pea`) khỏi thế giới.
    * Khởi tạo một viên đậu lửa (`FirePea`) với cùng dữ liệu hàng.
    * Thêm `FirePea` vào đúng vị trí của viên đạn cũ.



---

## 4. Cơ chế Sinh tồn và Phản hồi
* **Hệ thống Hit Flash:** Khi bị Zombie tấn công, `Torchwood` sử dụng hệ thống nháy trắng trên bộ ảnh `firewood` để báo hiệu mức độ nguy hiểm cho người chơi.
* **Tự giải phóng bộ nhớ:** Khi `hp <= 0`, đối tượng tự gọi `getWorld().removeObject(this)` để biến mất khỏi bàn cờ, đảm bảo logic trò chơi diễn ra chính xác.



---

## 5. Vai trò chiến thuật trong Hệ thống
`Torchwood` tạo ra một sự kết hợp (Synergy) mạnh mẽ trong Gameplay:
* **Đầu vào (Input):** Đối tượng `Pea` (Sát thương đơn).
* **Quá trình (Process):** Chuyển đổi thuộc tính thông qua va chạm vật lý.
* **Đầu ra (Output):** Đối tượng `FirePea` (Sát thương diện rộng/Sát thương x2).

---

## 6. Kết luận
Lớp `Torchwood` minh họa một cách xuất sắc kỹ thuật tương tác giữa các lớp đối tượng khác nhau. Việc sử dụng phương thức kiểm tra va chạm để thay thế một đối tượng bằng một phiên bản mạnh hơn là một giải pháp lập trình linh hoạt, giúp tăng tính chiến thuật và chiều sâu cho trò chơi.

\newpage

In [ ]:
import greenfoot.*;

public class TwinSunflower extends Plant
{
    private GreenfootImage[] idle;
    private boolean test = false;
    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;

    public TwinSunflower() {
        idle = importSprites("twinsunflower", 10);
        maxHp = 100;
        hp = maxHp;
    }

    public void update() {
        currentFrame = System.nanoTime();
        produceSun();
        animate(idle, 150, true);
    }

    public void hit(int dmg) {
        if (isLiving()) {
            hitFlash(idle, "twinsunflower");
            hp = hp - dmg;
        }
    }

    public void produceSun() {
        deltaTime2 = (currentFrame - lastFrame2) / 1000000;
        
        if (deltaTime2 > 20000L) {
            lastFrame2 = System.nanoTime();
            hitFlash(idle, "twinsunflower");
            test = true;
            
            
            PlayScene.addObject(new Sun(5), getX() + 20, getY() - 10);
            PlayScene.addObject(new Sun(5), getX() - 20, getY() - 10);
            PlayScene.addObject(new Sun(5), getX() - 20, getY() - 10);
        }
    }
}

# BÁO CÁO CẤU TRÚC LỚP TWINSUNFLOWER - GAME PROJECT
---

## 1. Giới thiệu chung
`TwinSunflower` là đơn vị sản xuất tài nguyên cao cấp trong trò chơi. Được phát triển từ nguyên mẫu của `Sunflower`, lớp này cho phép người chơi tạo ra lượng mặt trời gấp nhiều lần trong cùng một chu kỳ thời gian, giúp đẩy nhanh quá trình xây dựng đội quân thực vật.

---

## 2. Thông số kỹ thuật và Tài nguyên
Lớp này được tinh chỉnh để chịu được lượng sát thương cao hơn và có hiệu ứng hoạt ảnh chi tiết hơn:

| Thuộc tính | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **maxHp** | 100 | Sức chống chịu cao hơn phiên bản thường (60 HP). |
| **idle** | 10 Frames | Hoạt ảnh 10 khung hình giúp chuyển động mượt mà hơn. |
| **Chu kỳ sản xuất** | 20,000 ms | Khoảng cách giữa các đợt tạo tài nguyên (20 giây). |

---

## 3. Logic Sản xuất Tài nguyên Đa luồng
Điểm cốt lõi của `TwinSunflower` nằm ở hàm `produceSun()`. Thay vì tạo ra một đơn vị tài nguyên duy nhất, nó tạo ra một "đợt" mặt trời:

1. **Tính toán thời gian:** Tương tự lớp cha, sử dụng `System.nanoTime()` để đo lường `deltaTime2`.
2. **Cơ chế tạo tài nguyên:** Khi đủ 20 giây, hệ thống thực hiện lệnh `addObject()` liên tiếp 3 lần:
    * **Viên 1:** Lệch sang phải (`getX() + 20`).
    * **Viên 2 & 3:** Lệch sang trái (`getX() - 20`).
3. **Phản hồi thị giác:** Trước khi nhả mặt trời, cây sẽ thực hiện lệnh `hitFlash` để thông báo trạng thái sẵn sàng cho người chơi.



---

## 4. Quản lý Hoạt ảnh và Cập nhật
Hàm `update()` điều phối đồng thời hai tác vụ quan trọng:
* **Diễn họa:** `animate(idle, 150, true)` với tốc độ 150ms mỗi khung hình, nhanh hơn so với Sunflower thường (200ms), tạo cảm giác năng động hơn.
* **Logic:** Liên tục gọi `produceSun()` để kiểm tra điều kiện thời gian thực.

---

## 5. Vai trò chiến lược
`TwinSunflower` giải quyết bài toán không gian (Grid-space management). Bằng cách tập trung hỏa lực sản xuất vào một ô lưới:
* Giúp tiết kiệm diện tích trồng cây cho các đơn vị tấn công.
* Tăng cường khả năng hồi phục tài nguyên trong giai đoạn giữa và cuối trận đấu (Mid/Late Game).



---

## 6. Kết luận
Lớp `TwinSunflower` minh họa cho kỹ thuật mở rộng tính năng thông qua việc thay đổi các thông số đầu ra (Output Parameters). Bằng cách tùy biến tọa độ xuất hiện của vật thể tài nguyên, lớp này tạo ra một trải nghiệm gameplay phong phú và có chiều sâu chiến thuật hơn cho người chơi.

\newpage

In [ ]:
import greenfoot.*; 
public class Walnut extends Plant
{
    private GreenfootImage[] idle;
    private GreenfootImage[] d;
    private GreenfootImage[] dd;
    private boolean test = false;
    private long lastFrame2 = System.nanoTime();
    private long deltaTime2;
    public Walnut() {
        idle = importSprites("walnut", 5);
        d = importSprites("walnutd", 5);
        dd = importSprites("walnutdd", 5);
        maxHp = 600;
        hp = maxHp;
    }
    
    public void update(){
        if (hp >= 480) {
            animate(idle, 200, true);
        } else if (hp >= 240) {
            animate(d, 200, true);
        } else {
            animate(dd, 200, true);
        }

    }
    public void hit(int dmg) {
        if (isLiving()) {
            if (hp >= 480) {
                hitFlash(idle, "walnut");
            } else if (hp >= 240) {
                hitFlash(d, "walnutd");
            } else {
                hitFlash(dd, "walnutdd");
            }
            
            
            hp = hp-dmg;
        }
    }
  
}


# BÁO CÁO CẤU TRÚC LỚP WALNUT - GAME PROJECT
---

## 1. Giới thiệu chung
`Walnut` là đơn vị phòng thủ chủ lực trong trò chơi. Không giống các đơn vị khác tập trung vào tấn công hay sản xuất, `Walnut` được thiết kế với lượng máu cực lớn để trì hoãn sự tiến công của kẻ thù. Đặc điểm nổi bật của lớp này là hệ thống phản hồi thị giác thông qua các trạng thái hư hại của cơ thể.

---

## 2. Quản lý Tài nguyên và Trạng thái Hư hại
Để người chơi dễ dàng nhận biết tình trạng sức khỏe của hàng phòng thủ, `Walnut` quản lý 3 bộ Sprite khác nhau:

| Bộ Sprite | Tên biến | Ngưỡng máu (HP) | Mô tả thị giác |
| :--- | :--- | :--- | :--- |
| **Lành lặn** | `idle` | $HP \ge 480$ | Trạng thái nguyên vẹn, lắc lư bình thường. |
| **Nứt nhẹ** | `d` | $240 \le HP < 480$ | Bắt đầu xuất hiện các vết nứt trên vỏ. |
| **Hư hỏng nặng** | `dd` | $HP < 240$ | Vỏ bị vỡ nát nhiều, biểu cảm đau đớn. |

* **maxHp:** 600 (Gấp 10 lần so với các loại cây tấn công thông thường như Peashooter).

---

## 3. Thuật toán Cập nhật theo Ngưỡng (Threshold-based Update)
Hàm `update()` của `Walnut` thực hiện kiểm tra điều kiện liên tục để quyết định bộ hoạt ảnh nào sẽ được diễn họa:

1. **Kiểm tra mức máu:** Sử dụng cấu trúc `if - else if - else` để phân loại trạng thái hiện tại dựa trên biến `hp`.
2. **Diễn họa:** Gọi hàm `animate()` tương ứng với bộ Sprite của trạng thái đó. Điều này giúp nhân vật thay đổi ngoại hình ngay lập tức khi bị Zombie cắn vượt ngưỡng máu quy định.



---

## 4. Logic Phản hồi Va chạm (Hit Detection)
Phương thức `hit()` được ghi đè để đảm bảo hiệu ứng `hitFlash` (nháy trắng) luôn khớp với hình dạng hiện tại của cây:
* Nếu cây đang ở trạng thái nứt nhẹ (`d`), khi bị cắn, nó phải nháy trắng trên bộ ảnh `d`.
* Nếu dùng nhầm bộ ảnh `idle` để nháy trong khi cây đang nứt, sẽ gây ra lỗi giật hình (visual glitch).
* Mã nguồn xử lý điều này bằng cách kiểm tra lại ngưỡng máu một lần nữa trong hàm `hit()`.



---

## 5. Vai trò chiến thuật
`Walnut` đóng vai trò là "Tanker" trong hệ thống:
* **Thu hút sát thương:** Chịu đòn thay cho các cây yếu máu ở phía sau.
* **Tạo khoảng trống thời gian:** Giúp các cây tấn công tầm xa (`Peashooter`, `GatlingPea`) có đủ thời gian để tiêu diệt Zombie trước khi chúng tiếp cận hàng phòng ngự cuối cùng.

---

## 6. Kết luận
Lớp `Walnut` là minh chứng cho việc sử dụng đồ họa để biểu thị dữ liệu logic (`hp`). Thay vì dùng thanh máu (Health bar) truyền thống, việc thay đổi Sprite theo ngưỡng máu giúp trò chơi trông tự nhiên, sinh động và chuyên nghiệp hơn.

\newpage

In [ ]:
import greenfoot.*;
import java.util.ArrayList;
import java.util.List;

public class Projectile extends SpriteAnimator {
    private int speed;
    private boolean hit = false;
    private boolean foundTarget = false;
    private Zombie hitZombie;
    private int frameCount;
    private int yPos; 
    private int damage;
    private GreenfootImage[] image;

    public Projectile(String name, int frameCount, int yPos, int dmg, int speed) {
        this.frameCount = frameCount;
        this.image = importSprites(name, frameCount);
        this.yPos = yPos;
        this.damage = dmg;
        this.speed = speed;
        if (image != null && image.length > 0) {
            setImage(image[0]);
        }
    }

    public void act() {
        
        PlayScene world = (PlayScene) getWorld();
        if (world == null) return;

        if (!world.getObjects(Overlay.class).isEmpty()) {
            return; 
        }
        if (hit) {
            handleHitAnimation(world);
            return;
        }

        move(speed);

        if (isAtEdge()) {
            world.removeObject(this);
            return;
        }

        checkCollision(world);
    }

    private void handleHitAnimation(PlayScene world) {
        
        if (frame >= frameCount - 1) {
            world.removeObject(this);
        } else {
            animate(image, 150, false);
        }
    }

    private void checkCollision(PlayScene world) {
        
        if (world.level == null || world.level.zombieRow == null) return;

        if (yPos < 0 || yPos >= world.level.zombieRow.size()) {
            return; 
        }

        List<Zombie> row = world.level.zombieRow.get(yPos);
        if (row == null || row.isEmpty()) return;

        for (Zombie z : new ArrayList<>(row)) {
            if (getWorld() == null) return;
            if (z == null || z.getWorld() == null) continue;

            if (Math.abs(z.getX() - getX()) < 30) {
                if (!hit) {
                    z.hit(damage);
                    this.hit = true;
                    this.frame = 0;
                }
                break;
            }
        }
    }

    public int getYPos() {
        return this.yPos;
    }
}

# BÁO CÁO CẤU TRÚC LỚP PROJECTILE - GAME PROJECT
---

## 1. Giới thiệu chung
Lớp `Projectile` kế thừa từ `SpriteAnimator`, đóng vai trò là lớp nền cho mọi loại đạn bắn ra từ thực vật. Lớp này chịu trách nhiệm quản lý quỹ đạo bay, xử lý tương tác vật lý với kẻ thù và điều khiển chuỗi hoạt ảnh từ lúc bắt đầu bay cho đến khi va chạm và tan biến.

---

## 2. Các thuộc tính kỹ thuật
Cấu trúc lớp được thiết kế để linh hoạt cho nhiều loại đạn khác nhau:

| Thuộc tính | Kiểu dữ liệu | Mô tả |
| :--- | :--- | :--- |
| `speed` | `int` | Tốc độ di chuyển của viên đạn trên trục X. |
| `damage` | `int` | Lượng sát thương sẽ trừ vào HP của Zombie khi va chạm. |
| `yPos` | `int` | Chỉ số hàng (Row index) mà viên đạn đang bay trên đó. |
| `hit` | `boolean` | Trạng thái xác định đạn đã va chạm với mục tiêu hay chưa. |

---

## 3. Logic Di chuyển và Giới hạn Màn hình
Viên đạn sử dụng phương thức `move(speed)` kế thừa từ lớp vật lý để di chuyển theo chiều ngang. Hệ thống tự động kiểm tra biên để tối ưu hóa bộ nhớ:
* **Kiểm tra biên:** Sử dụng `isAtEdge()`. Nếu viên đạn bay ra khỏi màn hình mà không trúng bất kỳ mục tiêu nào, nó sẽ tự động bị xóa khỏi World để tránh rò rỉ bộ nhớ (Memory Leak).



---

## 4. Thuật toán Kiểm tra Va chạm (Collision Logic)
Thay vì sử dụng các hàm va chạm mặc định nặng nề, `Projectile` sử dụng giải thuật quét theo hàng để tăng hiệu suất:

1. **Xác định hàng:** Chỉ kiểm tra va chạm với danh sách Zombie thuộc hàng `yPos` tương ứng.
2. **Tính toán khoảng cách:** Duyệt qua danh sách Zombie trong hàng và tính khoảng cách tuyệt đối: 
   $$\Delta x = |x_{zombie} - x_{projectile}|$$
3. **Kích hoạt va chạm:** Nếu $\Delta x < 30$ pixels:
   * Gọi hàm `z.hit(damage)` để trừ máu Zombie.
   * Chuyển trạng thái `hit = true` để dừng di chuyển và bắt đầu diễn họa nổ.



---

## 5. Diễn họa hiệu ứng va chạm (Hit Animation)
Khi biến `hit` trở thành `true`, viên đạn dừng di chuyển và chuyển sang hàm `handleHitAnimation()`:
* **Animation một lần:** Chạy hoạt ảnh nổ (vỡ tan) dựa trên bộ ảnh `image`.
* **Kết thúc vòng đời:** Khi hoạt ảnh nổ đạt đến khung hình cuối cùng (`frame >= frameCount - 1`), đối tượng sẽ chính thức bị xóa khỏi thế giới game.

---

## 6. Kết luận
Lớp `Projectile` là một thành phần tối yếu trong hệ thống chiến đấu. Bằng cách kết hợp giữa logic di chuyển vật lý và kiểm tra va chạm tối ưu theo hàng, lớp này đảm bảo việc xử lý hàng chục viên đạn cùng lúc trên màn hình vẫn giữ được tốc độ khung hình ổn định, đồng thời cung cấp hiệu ứng thị giác sinh động khi tương tác với kẻ thù.

\newpage

In [ ]:
import greenfoot.*; 
public class FirePea extends Projectile
{
    public FirePea(int yPos) {
        super("firepea", 3, yPos, 30, 10);
        
    }
}


# BÁO CÁO CẤU TRÚC LỚP FIREPEA - GAME PROJECT
---

## 1. Giới thiệu chung
`FirePea` là một lớp con chuyên biệt kế thừa từ lớp `Projectile`. Đây là loại đạn cao cấp được tạo ra khi một viên đậu thường đi xuyên qua `Torchwood` hoặc được bắn ra trực tiếp từ `MegaGatlingPea`. Lớp này đại diện cho cơ chế nâng cấp sức mạnh đạn trong trò chơi.

---

## 2. Tính kế thừa và Khởi tạo (Inheritance & Constructor)
Điểm đặc biệt của lớp `FirePea` là sự tối giản trong mã nguồn nhờ tận dụng tối đa lớp cha. Hàm khởi tạo (`Constructor`) sử dụng từ khóa `super()` để định nghĩa các thông số riêng biệt:

| Tham số truyền vào | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **Tên Sprite** | `"firepea"` | Hệ thống sẽ tự nạp các ảnh `firepea1.png`, `firepea2.png`... |
| **Số khung hình** | 3 | Chuỗi hoạt ảnh khi đạn bay và nổ. |
| **Sát thương (Damage)** | 30 | Cao hơn đáng kể so với đạn đậu thường (thường là 20). |
| **Tốc độ (Speed)** | 10 | Đảm bảo tốc độ bay nhanh và ổn định trên hàng. |

---

## 3. Cơ chế hoạt động
Mặc dù lớp `FirePea` không chứa các hàm logic bên trong, nó hoạt động dựa trên các quy tắc đã được thiết lập sẵn ở lớp `Projectile`:

1.  **Di chuyển:** Tự động bay sang phải với tốc độ 10 đơn vị mỗi khung hình.
2.  **Va chạm:** Tự động kiểm tra khoảng cách với Zombie trên cùng hàng `yPos`.
3.  **Diễn họa:** Tự động chạy hoạt ảnh nổ của bộ ảnh "firepea" khi va chạm thành công.



---

## 4. Vai trò trong Hệ thống Chiến đấu
`FirePea` là kết quả của sự tương tác giữa các thực thể (Entity Interaction):
* **Sự kết hợp (Synergy):** Khi `Pea` gặp `Torchwood`, đối tượng cũ bị hủy và `FirePea` được sinh ra.
* **Hiệu quả:** Việc tách riêng `FirePea` thành một lớp giúp lập trình viên dễ dàng tùy chỉnh sát thương hoặc hiệu ứng hình ảnh (như thêm lửa, khói) mà không làm ảnh hưởng đến các loại đạn khác.

---

## 5. Kết luận
Lớp `FirePea` minh chứng cho sức mạnh của lập trình hướng đối tượng (OOP). Việc kế thừa từ `Projectile` giúp mã nguồn cực kỳ gọn gàng, dễ bảo trì và mở rộng, đồng thời tạo ra sự đa dạng trong trải nghiệm chiến thuật của trò chơi.

\newpage

In [ ]:
import greenfoot.*; 
public class Needle extends Projectile
{
    public Needle(int yPos) {
        super("needle", 7, yPos, 8, 6);
        
    }
}


# BÁO CÁO CẤU TRÚC LỚP NEEDLE - GAME PROJECT
---

## 1. Giới thiệu chung
`Needle` là một lớp đối tượng đại diện cho loại đạn Gai, được sử dụng duy nhất bởi thực vật `Cactus`. Tương tự như các loại đạn khác, `Needle` tận dụng sức mạnh của tính kế thừa từ lớp `Projectile` để xử lý các hành vi vật lý và va chạm trên bàn cờ.

---

## 2. Đặc điểm kỹ thuật và Khởi tạo
Mặc dù có cấu trúc mã nguồn đơn giản, các thông số của `Needle` được tinh chỉnh để phản ánh đặc tính riêng của vũ khí này:

| Tham số truyền vào | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **Tên Sprite** | `"needle"` | Hệ thống sẽ nạp chuỗi ảnh từ `needle1.png` đến `needle7.png`. |
| **Số khung hình** | 7 | Chu kỳ hoạt ảnh dài hơn giúp hiệu ứng gai bay và vỡ tan chi tiết hơn. |
| **Sát thương (Damage)** | 8 | Mức sát thương cơ bản cho mỗi lần xuyên thấu. |
| **Tốc độ (Speed)** | 6 | Tốc độ bay trung bình, tạo sự khác biệt về nhịp độ so với đạn Đậu. |

---

## 3. Logic Hoạt động thừa hưởng
Lớp `Needle` không cần định nghĩa lại các hàm vận hành mà sử dụng trực tiếp các giải thuật tối ưu từ lớp cha `Projectile`:

* **Xử lý hàng (Row Index):** Nhận tham số `yPos` từ `Cactus` để đảm bảo chỉ tương tác với các Zombie trên cùng một hàng ngang.
* **Va chạm hình học:** Tự động thực hiện phép tính khoảng cách tuyệt đối để xác định điểm va chạm với mục tiêu.
* **Quản lý vòng đời:** Tự động xóa đối tượng khi bay ra khỏi biên màn hình hoặc sau khi hoàn tất hoạt ảnh nổ (vỡ gai) khi trúng mục tiêu.



---

## 4. Vai trò trong Hệ thống Chiến đấu
`Needle` đóng vai trò quan trọng trong việc đa dạng hóa hỏa lực:
* **Phản hồi thị giác:** Hình dáng gai nhọn tạo ra sự khác biệt về mặt thẩm mỹ so với các loại đạn tròn (Pea), giúp người chơi dễ dàng nhận biết nguồn phát ra sát thương.
* **Tính tùy biến:** Bằng cách thay đổi giá trị trong hàm `super()`, lập trình viên có thể biến `Needle` thành một loại đạn xuyên thấu hoặc đạn có tốc độ cực cao mà không cần thay đổi cấu trúc lớp.

---

## 5. Kết luận
Lớp `Needle` là minh chứng cho sự linh hoạt của kiến trúc mã nguồn trong dự án. Việc kế thừa lớp `Projectile` giúp tiết kiệm thời gian phát triển, giảm thiểu lỗi logic và cho phép dễ dàng tạo ra nhiều loại đạn mới chỉ với một vài dòng lệnh khởi tạo đơn giản.

\newpage

In [ ]:
import greenfoot.*; 
public class Pea extends Projectile
{
    public Pea(int yPos) {
        super("pea", 3, yPos, 12, 6);
        
    }
}


# BÁO CÁO CẤU TRÚC LỚP PEA - GAME PROJECT
---

## 1. Giới thiệu chung
`Pea` là lớp đối tượng đại diện cho đạn Đậu thường, loại đạn phổ biến nhất trong trò chơi, được bắn ra từ các thực vật như `Peashooter` và `Repeater`. Lớp này tận dụng tối đa tính kế thừa từ `Projectile` để tối ưu hóa mã nguồn và đảm bảo tính nhất quán trong cơ chế chiến đấu.

---

## 2. Thông số kỹ thuật và Khởi tạo
Hàm khởi tạo (`Constructor`) của `Pea` định nghĩa các chỉ số cân bằng cho giai đoạn đầu của trò chơi:

| Tham số truyền vào | Giá trị | Ý nghĩa |
| :--- | :--- | :--- |
| **Tên Sprite** | `"pea"` | Hệ thống nạp chuỗi ảnh hiệu ứng đạn bay và vỡ từ `pea1.png` đến `pea3.png`. |
| **Số khung hình** | 3 | Chu kỳ hoạt ảnh ngắn, phù hợp với hiệu ứng va chạm nhanh. |
| **Sát thương (Damage)** | 12 | Mức sát thương tiêu chuẩn để tiêu diệt các Zombie cơ bản. |
| **Tốc độ (Speed)** | 6 | Tốc độ bay cơ sở, tạo nhịp độ chiến đấu ổn định. |

---

## 3. Cơ chế thừa hưởng và Vận hành
Bằng cách gọi `super()`, `Pea` thừa hưởng toàn bộ quy trình xử lý của lớp cha mà không cần viết thêm mã nguồn phức tạp:

* **Xác định quỹ đạo:** Nhận tham số `yPos` để tự động bay trên hàng mà thực vật đã bắn nó ra.
* **Va chạm thông minh:** Sử dụng thuật toán quét khoảng cách tuyệt đối trên hàng để trừ máu Zombie khi tiếp xúc.
* **Hiệu ứng nổ:** Khi biến `hit` ở lớp cha chuyển sang `true`, `Pea` sẽ tự động hiển thị hoạt ảnh vỡ tan trước khi bị xóa khỏi thế giới.



---

## 4. Vai trò trong Hệ thống Chiến đấu
`Pea` đóng vai trò là "vật liệu gốc" trong chuỗi nâng cấp sát thương:
* **Tính phổ biến:** Xuất hiện trong hầu hết các đợt tấn công từ thực vật tầm xa.
* **Khả năng tương tác:** Là đối tượng duy nhất có thể tương tác với `Torchwood` để biến đổi thành `FirePea`, tăng gấp đôi sát thương.
* **Hiệu suất:** Nhờ cấu trúc kế thừa đơn giản, hệ thống có thể quản lý hàng trăm đối tượng `Pea` cùng lúc mà không gây sụt giảm khung hình (FPS).



---

## 5. Kết luận
Lớp `Pea` là minh chứng cho thiết kế mã nguồn hiệu quả (Clean Code). Việc cô lập các thông số đặc trưng trong hàm khởi tạo và để lớp cha xử lý logic giúp hệ thống dễ dàng bảo trì, cân bằng chỉ số sát thương và mở rộng thêm nhiều loại đạn mới một cách nhanh chóng.